In [1]:
from preprocessing import build_sequences, split
from train import fine_tune
seqs = build_sequences('token')
train, test = split(seqs)
fine_tune('token', train, rank=4) 

c:\Users\rosli\miniconda3\envs\thesis\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[token] 6154 sequences (114 normal, 6040 anomalous)
  Train (normal only): 57
  Test  (mixed):       3077  (57 normal, 3020 anomalous)

  Loading base model...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 288/288 [00:08<00:00, 35.68it/s]


trainable params: 1,597,440 || all params: 2,615,939,328 || trainable%: 0.0611
  Preparing dataset (57 sequences)...


Map: 100%|██████████| 57/57 [00:00<00:00, 123.88 examples/s]
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


...

c:\Users\rosli\miniconda3\envs\thesis\Lib\site-packages\torch\_dynamo\eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
5,2.641545
10,2.478658


c:\Users\rosli\miniconda3\envs\thesis\Lib\site-packages\torch\_dynamo\eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
c:\Users\rosli\miniconda3\envs\thesis\Lib\site-packages\torch\_dynamo\eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*ar

  Saved adapter to checkpoints\token\r4


WindowsPath('checkpoints/token/r4')

In [2]:
from preprocessing import build_sequences, split
from score import load_finetuned, score_sequences
from evaluate import evaluate
#seqs = build_sequences('token')
#_, test = split(seqs)
model, tok = load_finetuned('token', rank=4)
scores, labels = score_sequences(test, model, tok)
evaluate(scores, labels, model_name='llm_lora', service='token')

Loading weights: 100%|██████████| 288/288 [00:09<00:00, 29.80it/s]


    Scored 10/3077  (last: 1.8756, label=1)
    Scored 20/3077  (last: 2.6962, label=1)
    Scored 30/3077  (last: 2.4106, label=1)
    Scored 40/3077  (last: 1.7127, label=1)
    Scored 50/3077  (last: 2.7614, label=1)
    Scored 60/3077  (last: 2.4672, label=1)
    Scored 70/3077  (last: 2.5623, label=1)
    Scored 80/3077  (last: 2.4512, label=1)
    Scored 90/3077  (last: 2.6278, label=1)
    Scored 100/3077  (last: 2.5114, label=1)
    Scored 110/3077  (last: 1.9830, label=1)
    Scored 120/3077  (last: 2.9213, label=1)
    Scored 130/3077  (last: 2.4815, label=1)
    Scored 140/3077  (last: 2.4342, label=1)
    Scored 150/3077  (last: 1.9815, label=1)
    Scored 160/3077  (last: 2.6943, label=1)
    Scored 170/3077  (last: 2.1367, label=1)
    Scored 180/3077  (last: 2.6407, label=1)
    Scored 190/3077  (last: 2.5622, label=1)
    Scored 200/3077  (last: 2.9654, label=1)
    Scored 210/3077  (last: 2.5574, label=1)
    Scored 220/3077  (last: 2.8220, label=1)
    Scored 230/3077

{'service': 'token',
 'model': 'llm_lora',
 'aucroc': 0.4963,
 'f1': 0.9907,
 'threshold': np.float64(0.832),
 'n_test': 3077,
 'n_anomaly': 3020}

In [ ]:
seqs

In [2]:
import torch, json
from preprocessing import build_sequences, split, SERVICES
from train import fine_tune, LORA_RANK
from score import load_finetuned, score_sequences
from evaluate import evaluate, save_results

In [ ]:
RESULTS = []

for service in SERVICES:
    print(f"\n{'='*55}\n  Service: {service}\n{'='*55}")

    # ── Data ─────────────────────────────────────────────────────────────────
    seqs = build_sequences(service)
    if not seqs:
        print("  No data — skipping.")
        continue
    train_seqs, test_seqs = split(seqs)

    # ── Isolation Forest baseline ─────────────────────────────────────────────
    print("\n  [Isolation Forest]")
    if_res = run_isolation_forest(train_seqs, test_seqs)
    if if_res:
        RESULTS.append({**if_res, "service": service, "model": "isolation_forest"})

    # ── LLM + LoRA ────────────────────────────────────────────────────────────
    print("\n  [LLM + LoRA]")
    fine_tune(service, train_seqs, rank=LORA_RANK)

    model, tokenizer = load_finetuned(service, rank=LORA_RANK)
    scores, labels   = score_sequences(test_seqs, model, tokenizer)
    llm_res          = evaluate(scores, labels,
                                model_name="llm_lora", service=service)
    if llm_res:
        RESULTS.append(llm_res)

    del model; torch.cuda.empty_cache()

# ── Save & display ────────────────────────────────────────────────────────────
save_results(RESULTS)